#Import Libraries

In [23]:
from google.colab import files
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_rel

#Upload and read EEG Excel file

In [24]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print("EEG dataset loaded successfully.")
display(df.head())

print("Columns in uploaded file:")
print(df.columns)

Saving EEG_PrimaryOutcomes.xlsx to EEG_PrimaryOutcomes (3).xlsx
EEG dataset loaded successfully.


,ParticipantID,Attention_Task_NoSmell,Attention_Task_Smell,Relax_Task_NoSmell,Relax_Task_Smell,CogLoad_Task_NoSmell,CogLoad_Task_Smell
0,P04,-0.168,0.181,0.210,-0.098,0.948,0.948
1,P05,0.041,0.057,-0.128,-0.032,0.949,0.940
2,P06,-0.344,-0.330,0.334,0.416,0.959,0.955
3,P07,0.118,0.267,-0.129,-0.127,0.951,0.949
4,P08,-0.116,0.037,0.121,0.054,0.940,0.938


Columns in uploaded file:
Index(['ParticipantID', 'Attention_Task_NoSmell', 'Attention_Task_Smell',
       'Relax_Task_NoSmell', 'Relax_Task_Smell', 'CogLoad_Task_NoSmell',
       'CogLoad_Task_Smell'],
      dtype='object')


#Convert EEG columns to numeric

In [25]:
eeg_columns = [
    "Attention_Task_NoSmell",
    "Attention_Task_Smell",
    "Relax_Task_NoSmell",
    "Relax_Task_Smell",
    "CogLoad_Task_NoSmell",
    "CogLoad_Task_Smell"
]

required_columns = ["ParticipantID"] + eeg_columns

#Check required columns

In [26]:
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("ERROR: Missing columns:")
    for col in missing_columns:
        print("-", col)
    raise ValueError("Please check your Excel column names.")
else:
    print("All required EEG columns are present.")

All required EEG columns are present.


#Convert to numeric

In [27]:
for col in eeg_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("EEG columns converted to numeric.")

EEG columns converted to numeric.


#Define measure Pairs

In [28]:
measure_pairs = [
    ("Attention during task", "Attention_Task_NoSmell", "Attention_Task_Smell"),
    ("Relaxation during task", "Relax_Task_NoSmell", "Relax_Task_Smell"),
    ("Cognitive load during task", "CogLoad_Task_NoSmell", "CogLoad_Task_Smell")
]

#Descriptive statistics

In [29]:
descriptive_results = []

for measure_name, no_smell_col, smell_col in measure_pairs:

    descriptive_results.append({
        "EEG_Measure": measure_name,
        "Condition": "No-Smell",
        "Mean": df[no_smell_col].mean(),
        "SD": df[no_smell_col].std(ddof=1)
    })

    descriptive_results.append({
        "EEG_Measure": measure_name,
        "Condition": "Smell",
        "Mean": df[smell_col].mean(),
        "SD": df[smell_col].std(ddof=1)
    })

descriptive_df = pd.DataFrame(descriptive_results)

descriptive_df["Mean"] = descriptive_df["Mean"].round(2)
descriptive_df["SD"] = descriptive_df["SD"].round(2)

display(descriptive_df)
descriptive_file = "EEG_DescriptiveStatistics.xlsx"

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

print(f"{descriptive_file} saved.")

,EEG_Measure,Condition,Mean,SD
0,Attention during task,No-Smell,-0.09,0.16
1,Attention during task,Smell,0.04,0.18
2,Relaxation during task,No-Smell,0.01,0.17
3,Relaxation during task,Smell,0.07,0.18
4,Cognitive load during task,No-Smell,0.95,0.01
5,Cognitive load during task,Smell,0.95,0.01


EEG_DescriptiveStatistics.xlsx saved.


#Shapiro–Wilk normality test

In [30]:
normality_results = []

for measure_name, no_smell_col, smell_col in [
    ("Attention during task", "Attention_Task_NoSmell", "Attention_Task_Smell"),
    ("Relaxation during task", "Relax_Task_NoSmell", "Relax_Task_Smell"),
    ("Cognitive load during task", "CogLoad_Task_NoSmell", "CogLoad_Task_Smell")
]:

    difference_scores = (
        df[smell_col] - df[no_smell_col]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "EEG_Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "EEG_Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

# Round final normality table for dissertation reporting
normality_df = normality_df.round({
    "Shapiro_W": 2,
    "p_value": 3
})

display(normality_df)

normality_file = "EEG_NormalityResults.xlsx" #Table F.2

normality_df.to_excel(
    normality_file,
    index=False
)

print(f"{normality_file} saved.")

,EEG_Measure,Shapiro_W,p_value,Normality_Assumption
0,Attention during task,0.95,0.504,Met
1,Relaxation during task,0.94,0.340,Met
2,Cognitive load during task,0.92,0.172,Met


EEG_NormalityResults.xlsx saved.


#Paired-samples t-tests and Cohen’s d

In [34]:
ttest_results = []

for measure_name, no_smell_col, smell_col in measure_pairs:

    paired_data = df[[no_smell_col, smell_col]].dropna()

    no_smell_values = paired_data[no_smell_col]
    smell_values = paired_data[smell_col]

    if len(paired_data) >= 2:

        # Paired-samples t-test
        t_stat, p_value = ttest_rel(smell_values, no_smell_values)

        # Cohen's d for paired samples
        difference_scores = smell_values - no_smell_values
        difference_sd = difference_scores.std(ddof=1)

        if difference_sd == 0:
            cohens_d = np.nan
        else:
            cohens_d = difference_scores.mean() / difference_sd

        ttest_results.append({
            "EEG_Measure": measure_name,
            "t_value": round(t_stat, 2),
            "p_value": round(p_value, 3),
            "Cohens_d": round(cohens_d, 2)
        })

    else:
        ttest_results.append({
            "EEG_Measure": measure_name,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "EEG_PairedTTestResults.xlsx"

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

print(f"{ttest_file} saved.")

,EEG_Measure,t_value,p_value,Cohens_d
0,Attention during task,3.77,0.002,0.91
1,Relaxation during task,1.62,0.124,0.39
2,Cognitive load during task,0.93,0.368,0.22


EEG_PairedTTestResults.xlsx saved.


#Download results

In [35]:
from google.colab import files

files.download("EEG_PrimaryOutcomes.xlsx")          # needed for master dataset
files.download("EEG_DescriptiveStatistics.xlsx")    # Table 5.4
files.download("EEG_NormalityResults.xlsx")         # Table F.2
files.download("EEG_PairedTTestResults.xlsx")       # Table 5.5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>